## 🎯 Learning Objectives
* Understand the concept and importance of conditional edges and routing logic in LangGraph.
* Implement a router function to dynamically control agent workflow based on state.
* Utilize `add_conditional_edges` in LangGraph's `MessageGraph` to create adaptive agent behaviors.
* Identify common use cases for conditional routing in building robust and intelligent AI agents.


## Adding Conditional Edges and Routing Logic in LangGraph

Imagine you're building a sophisticated AI assistant. Sometimes it needs to use a tool to fetch information, other times it needs to generate a direct answer, and occasionally it might need to ask for clarification. How does it decide which path to take at any given moment? This is where **conditional edges** and **routing logic** in LangGraph become indispensable.

At its core, LangGraph allows you to define a graph where nodes represent steps (like calling an LLM, using a tool, or performing a data transformation) and edges represent the flow between these steps. While direct edges provide a fixed sequence, conditional edges introduce dynamic decision-making, making your agent truly adaptive.

Think of a conditional edge as a **traffic controller** for your agent's workflow. Instead of a fixed road leading to the next intersection, the traffic controller (your *router function*) observes the current traffic conditions (the agent's `state`) and then directs vehicles (the workflow) down different paths based on predefined rules. This allows your agent to:

1.  **React Dynamically**: Respond differently based on the user's input, the results of previous steps, or internal flags.
2.  **Implement Complex Logic**: Build sophisticated decision trees, ReAct loops (Reasoning and Acting), and multi-tool agents where the agent intelligently selects the next action.
3.  **Handle Edge Cases**: Route to specific error handling nodes or human-in-the-loop interventions when necessary.

In LangGraph, this is achieved by defining a Python function, often called a **router**, that inspects the current `state` of the graph. This router function returns a string representing the name of the next node to execute. LangGraph then uses this return value to select the appropriate edge from a set of conditional edges you've defined.

For instance, in a typical ReAct agent, after an LLM generates a thought and an action, a router function would examine the LLM's output. If the LLM decided to `CALL_TOOL`, the router would direct the workflow to a `tool_node`. If the LLM decided to `FINAL_ANSWER`, the router would direct it to an `end_node` or a `response_generation_node`. This dynamic routing is the backbone of intelligent, goal-driven agents in 2026 and beyond, enabling them to navigate complex tasks with unprecedented flexibility.


In [ ]:
import operator
from typing import Annotated, List, Literal, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.graph import END, MessageGraph

# --- 1. Define the Agent State --- #
# The state for our agent will be a list of messages.
# This is a common pattern for conversational agents.
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# --- 2. Define the Nodes (Agent Steps) --- #

# Mock LLM function: Simulates an LLM's response.
# In a real scenario, this would call an actual LLM API.
def llm_node(state: AgentState) -> AgentState:
    print("---LLM Node: Generating response---")
    last_message = state["messages"][-1]
    
    # Simulate LLM deciding to call a tool or provide a final answer
    if "tool needed" in last_message.content.lower():
        # Simulate a tool call instruction from an LLM
        ai_message = AIMessage(content="Okay, I need to use a tool. CALL_TOOL")
    elif "hello" in last_message.content.lower():
        ai_message = AIMessage(content="Hello there! How can I assist you today?")
    else:
        # Simulate a final answer from an LLM
        ai_message = AIMessage(content="This is my final answer based on your query.")
        
    return {"messages": [ai_message]}

# Mock Tool function: Simulates a tool's execution.
# In a real scenario, this would execute an actual tool (e.g., search, calculator).
def tool_node(state: AgentState) -> AgentState:
    print("---Tool Node: Executing tool---")
    # Simulate tool execution and its output
    tool_output_message = AIMessage(content="Tool output: Data fetched successfully!")
    return {"messages": [tool_output_message]}

# --- 3. Define the Router Function --- #
# This function decides the next step based on the last message in the state.
def router(state: AgentState) -> Literal["tool", "llm", "end"]:
    print("---Router: Deciding next step---")
    last_message = state["messages"][-1]
    
    if isinstance(last_message, AIMessage) and "CALL_TOOL" in last_message.content:
        print("Router decision: Route to tool_node")
        return "tool"
    elif isinstance(last_message, AIMessage) and "Tool output" in last_message.content:
        # After a tool call, we typically want the LLM to process the output
        print("Router decision: Route back to llm_node to process tool output")
        return "llm"
    else:
        # If it's a HumanMessage or an AIMessage that's a final answer
        # we can decide to end or continue with LLM for further processing.
        # For this example, if it's not a tool call, we consider it an end or final LLM step.
        if isinstance(last_message, AIMessage) and "final answer" in last_message.content.lower():
            print("Router decision: Route to END (final answer)")
            return "end"
        else:
            # Default to LLM if no specific instruction, or if it's a new human message
            print("Router decision: Route to llm_node (default)")
            return "llm"

# --- 4. Build the LangGraph Graph --- #

# Initialize the MessageGraph with our state definition
workflow = MessageGraph(AgentState)

# Add nodes to the graph
workflow.add_node("llm", llm_node)
workflow.add_node("tool", tool_node)

# Set the entry point for the graph
workflow.set_entry_point("llm")

# Add conditional edges
# From 'llm' node, use the router to decide if it goes to 'tool' or 'end'
workflow.add_conditional_edges(
    "llm", # The node from which the conditional edge originates
    router, # The router function that makes the decision
    {
        "tool": "tool", # If router returns "tool", go to the "tool" node
        "end": END,     # If router returns "end", terminate the graph
        "llm": "llm"    # If router returns "llm", go back to the "llm" node (e.g., for processing tool output)
    }
)

# From 'tool' node, after tool execution, we always want to go back to the LLM
# to process the tool's output and decide the next step.
workflow.add_edge("tool", "llm")

# Compile the graph into a runnable agent
app = workflow.compile()

# --- 5. Run the Agent with Different Inputs --- #

print("\n--- Running Agent: Scenario 1 (Direct Answer) ---")
inputs_1 = {"messages": [HumanMessage(content="Hello, tell me about LangGraph.")]}
for s in app.stream(inputs_1):
    print(s)

print("\n--- Running Agent: Scenario 2 (Tool Needed) ---")
inputs_2 = {"messages": [HumanMessage(content="I need to find some data, a tool needed.")]}
for s in app.stream(inputs_2):
    print(s)

print("\n--- Running Agent: Scenario 3 (Tool Output Processed) ---")
# This scenario demonstrates the full cycle: LLM -> Tool -> LLM (to process output) -> END
# We'll simulate the initial LLM output that triggers a tool call.
inputs_3 = {"messages": [HumanMessage(content="Find me the latest market trends (tool needed).")]}
for s in app.stream(inputs_3):
    print(s)


### Interpreting the Code Output and Use Cases

The code demonstrates how conditional edges enable dynamic routing within a LangGraph agent. Let's break down the outputs:

*   **Scenario 1 (Direct Answer)**: The `HumanMessage` "Hello, tell me about LangGraph." is processed by the `llm_node`. Since the LLM's simulated response doesn't contain "CALL_TOOL" or "tool needed", the `router` function directs the flow to `END` (because the simulated LLM provides a "final answer"). The graph executes `llm_node` once and then terminates.

*   **Scenario 2 (Tool Needed)**: The `HumanMessage` "I need to find some data, a tool needed." triggers the `llm_node`. This time, the simulated LLM's response *does* contain "CALL_TOOL". The `router` function intercepts this and, based on its logic, directs the flow to the `tool_node`. After the `tool_node` executes, the `add_edge("tool", "llm")` ensures the flow returns to the `llm_node` to process the tool's output. The `llm_node` then provides a final answer, and the router directs to `END`.

This output clearly illustrates how the `router` function, in conjunction with `add_conditional_edges`, acts as the decision-maker, allowing the agent to choose different execution paths based on the content of the `AgentState` (specifically, the last message).

#### Performance Trade-offs

Conditional edges introduce a slight overhead compared to fixed edges, as the router function must be executed at each decision point. However, this overhead is typically negligible for most applications, especially when compared to the latency of LLM calls or external tool executions. The primary performance consideration lies in the complexity of your `router` function. A highly complex router with extensive state analysis or external lookups could introduce noticeable delays. For optimal performance, keep router functions focused, efficient, and deterministic.

#### Typical Use Cases in 2026

Conditional edges and routing are fundamental for building advanced AI agents:

1.  **ReAct Agents**: The most common pattern, where an agent decides between `tool_use` and `final_answer` based on LLM output.
2.  **Multi-Tool Agents**: Agents that can choose from a suite of tools, routing to the specific tool node based on the user's query and LLM's reasoning.
3.  **Human-in-the-Loop Workflows**: Routing to a human review node if the agent detects uncertainty, requires approval, or encounters a sensitive query.
4.  **Error Handling and Recovery**: If a tool fails or an LLM generates an invalid response, the router can direct the flow to a dedicated error handling node or a retry mechanism.
5.  **Dynamic Sub-Agent Invocation**: Routing to different specialized sub-agents based on the task at hand (e.g., a 'research agent', a 'coding agent', a 'summarization agent').
6.  **Contextual Adaptation**: Agents that dynamically adjust their behavior or retrieve different information based on the conversation history or external context (e.g., user preferences, time of day).

As AI systems become more autonomous and integrated into complex business processes, the ability to define flexible, state-dependent workflows via conditional routing will be a cornerstone of robust and intelligent agent design.


### Resources

*   **LangGraph Official Documentation - Conditional Edges**: [https://langchain-ai.github.io/langgraph/how-to/conditional-edges/](https://langchain-ai.github.io/langgraph/how-to/conditional-edges/)
*   **LangGraph Official Documentation - MessageGraph**: [https://langchain-ai.github.io/langgraph/how-to/message-graph/](https://langchain-ai.github.io/langgraph/how-to/message-graph/)
*   **LangChain Blog - Building a ReAct Agent with LangGraph**: [https://blog.langchain.dev/langgraph-for-building-agents/](https://blog.langchain.dev/langgraph-for-building-agents/)
*   **LangChain Core Messages**: [https://api.python.langchain.com/en/latest/messages/langchain_core.messages.html](https://api.python.langchain.com/en/latest/messages/langchain_core.messages.html)
